# Clustering Benchmark Results Summary

Load all CSV results from Python and R benchmarks, compute mean (std) for each metric.

In [1]:
import pandas as pd
import numpy as np
import os

## 1. Load All Result Files

In [2]:
# Define all result files and their metadata
# Format: (filename, method, language, platform)
result_files = [
    # Python CPU + GPU
    ("python_kmeans_results.csv",  "K-Means",  "Python", "cpu_gpu"),
    ("python_hdbscan_results.csv", "HDBSCAN",  "Python", "cpu_gpu"),
    ("python_dbscan_results.csv",  "DBSCAN",   "Python", "cpu_gpu"),
    ("python_hc_results.csv",      "HC",       "Python", "cpu_gpu"),
    # Python CPU only (GMM has no cuML support)
    ("python_gmm_results.csv",     "GMM",      "Python", "cpu_only"),
    # R
    ("R_kmeans_results.csv",  "K-Means",  "R", "r"),
    ("R_hdbscan_results.csv", "HDBSCAN",  "R", "r"),
    ("R_dbscan_results.csv",  "DBSCAN",   "R", "r"),
    ("R_hc_results.csv",      "HC",       "R", "r"),
    ("R_gmm_results.csv",     "GMM",      "R", "r"),
]

# Load all available files
raw_data = {}
for fname, method, lang, platform in result_files:
    if os.path.exists(fname):
        raw_data[(method, lang, platform)] = pd.read_csv(fname)
        print(f"Loaded {fname} ({len(raw_data[(method, lang, platform)])} replicates)")
    else:
        print(f"MISSING: {fname}")

Loaded python_kmeans_results.csv (100 replicates)
Loaded python_hdbscan_results.csv (100 replicates)
Loaded python_dbscan_results.csv (100 replicates)
Loaded python_hc_results.csv (100 replicates)
Loaded python_gmm_results.csv (100 replicates)
Loaded R_kmeans_results.csv (100 replicates)
Loaded R_hdbscan_results.csv (100 replicates)
Loaded R_dbscan_results.csv (100 replicates)
Loaded R_hc_results.csv (100 replicates)
Loaded R_gmm_results.csv (100 replicates)


## 2. Compute Summary Statistics

Format: `mean (std)`

In [10]:
def fmt(values, decimals=4):
    """Format as 'mean (std)'"""
    return f"{np.mean(values):.{decimals}f} ({np.std(values):.{decimals}f})"

# Build summary rows
rows = []

for (method, lang, platform), df in raw_data.items():
    if platform == "cpu_gpu":
        # Python with both CPU and GPU results
        rows.append({
            "Method": method,
            "Implementation": f"sklearn",
            "Runtime": fmt(df["CPU_Runtime_sec"]),
            "ARI": fmt(df["CPU_ARI"], 4),
            "NMI": fmt(df["CPU_NMI"], 4),
        })
        rows.append({
            "Method": method,
            "Implementation": f"cuML",
            "Runtime": fmt(df["GPU_Runtime_sec"]),
            "ARI": fmt(df["GPU_ARI"], 4),
            "NMI": fmt(df["GPU_NMI"], 4),
        })
    elif platform == "cpu_only":
        # Python CPU only (e.g., GMM)
        rows.append({
            "Method": method,
            "Implementation": f"sklearn",
            "Runtime": fmt(df["CPU_Runtime_sec"]),
            "ARI": fmt(df["CPU_ARI"], 4),
            "NMI": fmt(df["CPU_NMI"], 4),
        })
    elif platform == "r":
        # R results
        rows.append({
            "Method": method,
            "Implementation": f"R",
            "Runtime": fmt(df["R_Runtime_sec"]),
            "ARI": fmt(df["R_ARI"], 4),
            "NMI": fmt(df["R_NMI"], 4),
        })

summary_df = pd.DataFrame(rows)
summary_df

,Method,Implementation,Runtime,ARI,NMI
0,K-Means,sklearn,0.0036 (0.0003),1.0000 (0.0000),1.0000 (0.0000)
1,K-Means,cuML,0.0017 (0.0003),1.0000 (0.0000),1.0000 (0.0000)
2,HDBSCAN,sklearn,1.2867 (0.0131),1.0000 (0.0000),1.0000 (0.0000)
3,HDBSCAN,cuML,0.0206 (0.0008),1.0000 (0.0000),1.0000 (0.0000)
4,DBSCAN,sklearn,0.4629 (0.2950),0.9000 (0.3000),0.9000 (0.3000)
5,DBSCAN,cuML,0.0078 (0.0010),0.9000 (0.3000),0.9000 (0.3000)
6,HC,sklearn,3.5672 (0.0614),1.0000 (0.0000),1.0000 (0.0000)
7,HC,cuML,0.0134 (0.0003),1.0000 (0.0000),1.0000 (0.0000)
8,GMM,sklearn,0.0130 (0.0057),1.0000 (0.0000),1.0000 (0.0000)
9,K-Means,R,0.0073 (0.0089),1.0000 (0.0000),1.0000 (0.0000)


## 3. Pivot Table by Method

In [11]:
method_order = ["K-Means", "GMM", "HDBSCAN", "DBSCAN", "HC"]
impl_order   = ["R", "sklearn", "cuML"]

summary_df["Method"] = pd.Categorical(summary_df["Method"], categories=method_order, ordered=True)
summary_df["Implementation"] = pd.Categorical(summary_df["Implementation"], categories=impl_order, ordered=True)

summary_df = summary_df.sort_values(["Method", "Implementation"]).reset_index(drop=True)
summary_df

,Method,Implementation,Runtime,ARI,NMI
0,K-Means,R,0.0073 (0.0089),1.0000 (0.0000),1.0000 (0.0000)
1,K-Means,sklearn,0.0036 (0.0003),1.0000 (0.0000),1.0000 (0.0000)
2,K-Means,cuML,0.0017 (0.0003),1.0000 (0.0000),1.0000 (0.0000)
3,GMM,R,0.3294 (0.1590),1.0000 (0.0000),1.0000 (0.0000)
4,GMM,sklearn,0.0130 (0.0057),1.0000 (0.0000),1.0000 (0.0000)
5,HDBSCAN,R,4.8100 (0.1280),1.0000 (0.0000),1.0000 (0.0000)
6,HDBSCAN,sklearn,1.2867 (0.0131),1.0000 (0.0000),1.0000 (0.0000)
7,HDBSCAN,cuML,0.0206 (0.0008),1.0000 (0.0000),1.0000 (0.0000)
8,DBSCAN,R,1.9498 (0.4316),0.9000 (0.3000),0.9000 (0.3000)
9,DBSCAN,sklearn,0.4629 (0.2950),0.9000 (0.3000),0.9000 (0.3000)


## 4. GPU Speedup Table

In [12]:
# Compute GPU speedup for methods that have both CPU and GPU
speedup_rows = []

for (method, lang, platform), df in raw_data.items():
    if platform == "cpu_gpu":
        cpu_mean = np.mean(df["CPU_Runtime_sec"])
        gpu_mean = np.mean(df["GPU_Runtime_sec"])
        speedup = cpu_mean / gpu_mean
        speedup_rows.append({
            "Method": method,
            "CPU Mean (s)": f"{cpu_mean:.4f}",
            "GPU Mean (s)": f"{gpu_mean:.4f}",
            "Speedup": f"{speedup:.2f}x",
        })

speedup_df = pd.DataFrame(speedup_rows)
if len(speedup_df) > 0:
    speedup_df["Method"] = pd.Categorical(speedup_df["Method"], categories=method_order, ordered=True)
    speedup_df = speedup_df.sort_values("Method").reset_index(drop=True)
speedup_df

,Method,CPU Mean (s),GPU Mean (s),Speedup
0,K-Means,0.0036,0.0017,2.06x
1,HDBSCAN,1.2867,0.0206,62.48x
2,DBSCAN,0.4629,0.0078,59.07x
3,HC,3.5672,0.0134,266.35x


## 5. Export to LaTeX

In [13]:
latex_table = summary_df.to_latex(
    index=False, 
    caption="Clustering Benchmark Results: Mean (Std) over 100 Replicates", 
    label="tab:clustering_results",
    column_format="llcc",  # Adjust alignment (l=left, c=center)
    escape=False           # Useful if your data contains special LaTeX characters like underscores
)

print(latex_table)

\begin{table}
\caption{Clustering Benchmark Results: Mean (Std) over 100 Replicates}
\label{tab:clustering_results}
\begin{tabular}{llcc}
\toprule
Method & Implementation & Runtime & ARI & NMI \\
\midrule
K-Means & R & 0.0073 (0.0089) & 1.0000 (0.0000) & 1.0000 (0.0000) \\
K-Means & sklearn & 0.0036 (0.0003) & 1.0000 (0.0000) & 1.0000 (0.0000) \\
K-Means & cuML & 0.0017 (0.0003) & 1.0000 (0.0000) & 1.0000 (0.0000) \\
GMM & R & 0.3294 (0.1590) & 1.0000 (0.0000) & 1.0000 (0.0000) \\
GMM & sklearn & 0.0130 (0.0057) & 1.0000 (0.0000) & 1.0000 (0.0000) \\
HDBSCAN & R & 4.8100 (0.1280) & 1.0000 (0.0000) & 1.0000 (0.0000) \\
HDBSCAN & sklearn & 1.2867 (0.0131) & 1.0000 (0.0000) & 1.0000 (0.0000) \\
HDBSCAN & cuML & 0.0206 (0.0008) & 1.0000 (0.0000) & 1.0000 (0.0000) \\
DBSCAN & R & 1.9498 (0.4316) & 0.9000 (0.3000) & 0.9000 (0.3000) \\
DBSCAN & sklearn & 0.4629 (0.2950) & 0.9000 (0.3000) & 0.9000 (0.3000) \\
DBSCAN & cuML & 0.0078 (0.0010) & 0.9000 (0.3000) & 0.9000 (0.3000) \\
HC & R & 6.110

In [7]:
# Save summary to CSV
summary_df.to_csv("summary_results.csv", index=False)
print("Saved to summary_results.csv")

Saved to summary_results.csv
